In [40]:
from typing import Callable, Tuple
from qiskit import QuantumCircuit
import qiskit
from qiskit import transpile
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Operator
import math
from math import pi
import numpy as np

from sqlalchemy.orm import joinedload
from sqlalchemy import select

from benchmarklib import BenchmarkDatabase
from benchmarklib.runners import BatchQueue
from benchmarklib.pipeline.synthesis import XAGSynthesizer, TruthTableSynthesizer

from experiments import ExperimentProblem, ExperimentTrial

TAG = "cmps_comparison"

In [41]:
service = QiskitRuntimeService()
backend = service.backend("ibm_rensselaer")
db = BenchmarkDatabase("experiments.db", ExperimentProblem, ExperimentTrial)

In [42]:
def generate_log_n_cmp(N):
    #flag[i] = 1 means i-th bit gives an answer 
    #initialize flag and result lists
    #flag = [a^b for (a,b) in zip(A,B)]
    res = []
    #res[i] is B[i] if bits are different; 0 otherwise
    #for c in range(N):
    #    res.append(flag[c] & B[c])
    #printing initialization for unrolled/linear circuit   
    output = []  
    output.append("def verify(inpt: Tuple[bool]) -> bool:")
    for c in range(N):
        output.append("flag_0_"+str(c)+" = inpt["+str(c)+"] ^ inpt["+str(N+c)+"]")
        output.append("res_0_"+str(c)+" = flag_0_"+str(c)+" & inpt["+str(N+c)+"]")
    step = 1
    while step <= N/2:
        i = 0
        while i < N:
            #if flag[i] is one, then res[i] holds comparison result
            #res[i] = MUX(flag[i],res[i],res[i+step])
            #if flag[i] is one, then keep the flag value, otherwise set to flag of i+step
            #flag[i] = MUX(flag[i],flag[i],flag[i+step]
            #res[i] = (flag[i] & res[i]) ^ ((not flag[i]) & res[i+step])
            #flag[i] = flag[i] ^ ((not flag[i]) & flag[i+step])
            output.append("t1_"+str(step)+"_"+str(i)+" = flag_"+str(step//2)+"_"+str(i)+" & res_"+str(step//2)+"_"+str(i))
            output.append("t2_"+str(step)+"_"+str(i)+" = (not flag_"+str(step//2)+"_"+str(i)+") & res_"+str(step//2)+"_"+str(i+step))
            output.append("res_"+str(step)+"_"+str(i)+" = t1_"+str(step)+"_"+str(i)+" ^ t2_"+str(step)+"_"+str(i))
            output.append("t4_"+str(step)+"_"+str(i)+" = (not flag_"+str(step//2)+"_"+str(i)+") & flag_"+str(step//2)+"_"+str(i+step))
            output.append("flag_"+str(step)+"_"+str(i)+" = flag_"+str(step//2)+"_"+str(i)+" ^ t4_"+str(step)+"_"+str(i))
            i = i + step*2
        step = step*2
    output.append("return res_"+str(int(N/2))+"_0")
    return '\n    '.join(output)   


def generate_seq_cmp(N):
    xor = []
    output = []
    output.append("def verify(inpt: Tuple[bool]) -> bool:")
    for c in range(N):
        #xor.append(A[c]^B[c])
        output.append("xor_"+str(c)+" = inpt["+str(c)+"] ^ inpt["+str(N+c)+"]")
    res = False
    flag = False
    output.append("res_0 = False")
    output.append("flag_0 = False")
    
    for i in range(0,N):
        #c = xor[i] & (not flag)
        output.append("c_"+str(i)+" = xor_"+str(i)+" & (not flag_"+str(i)+")")
        #flag =(c & True) ^ ((not c) & flag)
        output.append("t2_"+str(i)+" = (not c_"+str(i)+") & flag_"+str(i))
        output.append("flag_"+str(i+1)+" = c_"+str(i)+" ^ t2_"+str(i)) 
        #res = (c & B[i]) ^ ((not c) & res) # res2 = MUX(c,B[i],res1)
        output.append("t3_"+str(i)+" = c_"+str(i)+" & inpt["+str(N+i)+"]")
        output.append("t4_"+str(i)+" = (not c_"+str(i)+") & res_"+str(i))
        output.append("res_"+str(i+1)+" = t3_"+str(i)+" ^ t4_"+str(i))
    output.append("return res_"+str(N))
    return '\n    '.join(output)   

In [ ]:
# build the circuits we are comparing
problems = {
    "log_lt4": db.get_or_create(ExperimentProblem, name="log_lt4", tag=TAG, n=2*4),
    "seq_lt4": db.get_or_create(ExperimentProblem, name="seq_lt4", tag=TAG, n=2*4),
    "log_lt8": db.get_or_create(ExperimentProblem, name="log_lt8", tag=TAG, n=2*8),
    "seq_lt8": db.get_or_create(ExperimentProblem, name="seq_lt8", tag=TAG, n=2*8),
}

problems["log_lt4"].verifier_src = generate_log_n_cmp(4)
problems["seq_lt4"].verifier_src = generate_seq_cmp(4)
problems["log_lt8"].verifier_src = generate_log_n_cmp(8)
problems["seq_lt8"].verifier_src = generate_seq_cmp(8)

candidates = list(problems.keys())

base_circuits = {
    name : XAGSynthesizer().synthesize(problems[name]) for name in candidates
}

In [73]:
print("Pre-transpile Metrics")
for candidate in candidates:
    circuit = base_circuits[candidate]
    print(f"{candidate}: Depth: {circuit.depth()}; ops: {circuit.count_ops()}")
    

Pre-transpile Metrics
log_lt4: Depth: 16; ops: OrderedDict([('cx', 31), ('ccrx', 4), ('ccrx_o0', 2), ('ccrx_o2', 2), ('ccx', 1), ('ccx_o2', 1)])
seq_lt4: Depth: 42; ops: OrderedDict([('cx', 57), ('ccrx_o2', 8), ('ccrx', 4), ('ccx', 1)])
log_lt8: Depth: 32; ops: OrderedDict([('cx', 89), ('ccrx', 14), ('ccrx_o2', 10), ('ccrx_o0', 4), ('ccx_o2', 1)])
seq_lt8: Depth: 164; ops: OrderedDict([('cx', 245), ('ccrx_o2', 16), ('ccrx', 12), ('ccx', 1)])


In [47]:
def create_trials(problem: ExperimentProblem):
    trials = []
    N = problem.n // 2
    # generate a variety of input states to test with
    input_states = []
    for i in range(16):
        random_val = np.random.randint(0, 2**(2*N) - 1)
        input_state = format(random_val, '0' + str(2*N) + 'b')
        input_states.append(input_state)

    circuit_width = max(problem.n + 1, base_circuits[problem.name].width())
    
    for input_state in input_states:
        
        qc = QuantumCircuit(circuit_width, problem.n + 1)
        for qubit in range(problem.n):
            if input_state[qubit] == '1':
                qc.x(qubit)

        qc.compose(base_circuits[problem.name], inplace=True)
        qc.measure(range(problem.n + 1), range(problem.n + 1))

        qc_final = transpile(qc, backend=backend, optimization_level=3)

        # calculate expected output state
        result = "1" if input_state[0:N] < input_state[N:2*N] else "0"
        expected_output = result + input_state[::-1]
        
        trials.append(
            ExperimentTrial(
                problem=problem,
                circuit=qc_final,
                circuit_pretranspile=qc,
                extra_data={"input_state": input_state, "backend": backend.name, "expected_output": expected_output}
            )
        )
    return trials

In [48]:
# sanity check one of the trials
trial = create_trials(problems["log_lt4"])[0]
simulator = AerSimulator()
results = simulator.run(trial.circuit, shots=4096).result()
counts = results.get_counts()
print("Input State:", trial.extra_data["input_state"])
print("Expected Output:", trial.extra_data["expected_output"])
print("Counts:", counts)

Input State: 10111100
Expected Output: 100111101
Counts: {'100111101': 4096}


In [74]:
for candidate in candidates:
    problem = problems[candidate]
    trials = create_trials(problem)
    print(f"{candidate}: Depth: {trials[0].circuit.depth()}; ops: {trials[0].circuit.count_ops()}")

log_lt4: Depth: 318; ops: OrderedDict([('rz', 483), ('sx', 285), ('ecr', 169), ('x', 30), ('measure', 9)])
seq_lt4: Depth: 895; ops: OrderedDict([('rz', 1040), ('sx', 575), ('ecr', 372), ('x', 70), ('measure', 9)])
log_lt8: Depth: 797; ops: OrderedDict([('rz', 1726), ('sx', 1013), ('ecr', 613), ('x', 90), ('measure', 17)])
seq_lt8: Depth: 3496; ops: OrderedDict([('rz', 5311), ('sx', 2917), ('ecr', 1934), ('x', 342), ('measure', 17)])


In [64]:
with BatchQueue(db, backend=backend, shots=4096) as q:
    for candidate in candidates:
        problem = problems[candidate]
        trials = create_trials(problem)
        for trial in trials:
            q.enqueue(trial, trial.circuit, run_simulation=False)

In [60]:
await db.update_all_pending_results(service=service)

### Visualize Results

In [65]:
def calculate_fidelity(trial):
    expected_output = trial.extra_data["expected_output"]
    total_shots = sum(trial.counts.values())
    correct_shots = trial.counts.get(expected_output, 0)
    fidelity = correct_shots / total_shots
    return fidelity

In [66]:
trials = db.query(
    select(ExperimentTrial).join(ExperimentTrial.problem).options(joinedload(ExperimentTrial.problem))
    .where(ExperimentProblem.tag == TAG)
)

In [67]:
fidelities = {candidate: [] for candidate in candidates}
for trial in trials:
    fidelity = calculate_fidelity(trial)
    fidelities[trial.problem.name].append(fidelity)

In [68]:
for candidate in candidates:
    avg_fidelity = sum(fidelities[candidate]) / len(fidelities[candidate])
    print(f"{candidate}: Average Fidelity = {avg_fidelity:.5f}")

log_lt4: Average Fidelity = 0.00560
seq_lt4: Average Fidelity = 0.00231
log_lt8: Average Fidelity = 0.00002
seq_lt8: Average Fidelity = 0.00001


In [69]:
fidelities_positive = {candidate: [] for candidate in candidates}
fidelities_negative = {candidate: [] for candidate in candidates}
for trial in trials:
    fidelity = calculate_fidelity(trial)
    if trial.extra_data["expected_output"][0] == '1':
        fidelities_positive[trial.problem.name].append(fidelity)
    else:
        fidelities_negative[trial.problem.name].append(fidelity)

In [70]:
for candidate in candidates:
    avg_pos_fidelity = sum(fidelities_positive[candidate]) / len(fidelities_positive[candidate])
    avg_neg_fidelity = sum(fidelities_negative[candidate]) / len(fidelities_negative[candidate])
    print(f"{candidate}: Fidelity (Average Positive) = {avg_pos_fidelity:.8f}, Fidelity (Average Negative) = {avg_neg_fidelity:.8f}")

log_lt4: Fidelity (Average Positive) = 0.00507270, Fidelity (Average Negative) = 0.00598474
seq_lt4: Fidelity (Average Positive) = 0.00248718, Fidelity (Average Negative) = 0.00212860
log_lt8: Fidelity (Average Positive) = 0.00001395, Fidelity (Average Negative) = 0.00002526
seq_lt8: Fidelity (Average Positive) = 0.00000678, Fidelity (Average Negative) = 0.00000872
